In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


In [2]:
!ls /content/drive/MyDrive/masks2/masks_train2

CNV  DRUSEN  NORMAL


In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms, datasets, models
from tqdm import tqdm
import os


In [4]:
from PIL import Image, UnidentifiedImageError
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as transforms
import os

# === Custom Dataset care filtrează fișierele corupte ===
class OCTDataset(Dataset):
    def __init__(self, root_dir, transform=None, max_per_class=6000):
        self.transform = transform
        self.data = []
        self.classes = sorted(os.listdir(root_dir))
        self.class_to_idx = {cls_name: idx for idx, cls_name in enumerate(self.classes)}
        class_counts = {cls: 0 for cls in self.classes}

        print("Scanning folders and filtering valid images...")
        for cls in self.classes:
            cls_dir = os.path.join(root_dir, cls)
            if not os.path.isdir(cls_dir):
                continue
            for file in os.listdir(cls_dir):
                if class_counts[cls] >= max_per_class:
                    break
                file_path = os.path.join(cls_dir, file)
                try:
                    img = Image.open(file_path)
                    img.verify()  # verifică dacă fișierul e imagine validă
                    self.data.append((file_path, self.class_to_idx[cls]))
                    class_counts[cls] += 1
                except (UnidentifiedImageError, IOError):
                    print(f"Ignoring invalid image: {file_path}")

        print("Valid images per class:")
        for cls in self.classes:
            print(f"{cls}: {class_counts[cls]} images")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        path, label = self.data[idx]
        image = Image.open(path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label

# === Transformări standard pentru model ===
transform = transforms.Compose([
    transforms.Resize((160, 160)),
    transforms.ToTensor(),
])

# === Creăm dataset-ul ===
dataset_path = "/content/drive/MyDrive/masks2/masks_train2"
dataset = OCTDataset(dataset_path, transform=transform, max_per_class=6000)

# === Split train/val ===
val_split = 0.2
n_total = len(dataset)
n_val = int(n_total * val_split)
n_train = n_total - n_val

train_dataset, val_dataset = random_split(dataset, [n_train, n_val])

# === DataLoader pentru training rapid ===
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=4, pin_memory=True)

print(f"Total training images: {len(train_dataset)}")
print(f"Total validation images: {len(val_dataset)}")


Scanning folders and filtering valid images...
Valid images per class:
CNV: 2000 images
DRUSEN: 2000 images
NORMAL: 2000 images
Total training images: 4800
Total validation images: 1200


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


In [6]:
import os

dataset_path = "/content/drive/MyDrive/masks2/masks_train2"
classes = sorted([d for d in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, d))])
print("Classes:", classes)


Classes: ['CNV', 'DRUSEN', 'NORMAL']


In [8]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


In [9]:
model = models.mobilenet_v2(pretrained=True)
model.classifier[1] = nn.Linear(model.last_channel, len(classes))  # output 3 clase
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# Mixed Precision
scaler = torch.cuda.amp.GradScaler()

/tmp/ipython-input-3534513938.py:9: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


In [10]:
num_epochs = 30  # crește dacă vrei

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}/{num_epochs}")

    # --- Training ---
    model.train()
    train_loss = 0
    correct = 0
    total = 0
    for images, labels in tqdm(train_loader):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    print(f"Train Loss: {train_loss/total:.4f} | Train Acc: {correct/total:.4f}")

    # --- Validation ---
    model.eval()
    val_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in tqdm(val_loader):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    print(f"Val Loss: {val_loss/total:.4f} | Val Acc: {correct/total:.4f}")


Epoch 1/30


  0%|          | 0/150 [00:00<?, ?it/s]/tmp/ipython-input-4063459362.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
100%|██████████| 150/150 [00:50<00:00,  2.98it/s]


Train Loss: 0.4534 | Train Acc: 0.8058


100%|██████████| 38/38 [00:09<00:00,  4.21it/s]


Val Loss: 0.3051 | Val Acc: 0.8833

Epoch 2/30


...

Val Loss: 0.4560 | Val Acc: 0.9183

Epoch 30/30


100%|██████████| 150/150 [00:40<00:00,  3.71it/s]


Train Loss: 0.0155 | Train Acc: 0.9954


100%|██████████| 38/38 [00:08<00:00,  4.47it/s]

Val Loss: 0.5185 | Val Acc: 0.9108


In [11]:
torch.save(model.state_dict(), "/content/drive/MyDrive/oct_modelvar2_1_mobilnetv2.pth")

In [12]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, 3)  # 3 clase: CNV, DRUSEN, NORMAL
model = model.to(device)

# === Loss și optimizer ===
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# === AMP scaler pentru mixed precision ===
scaler = torch.cuda.amp.GradScaler()

# === Training loop ===
num_epochs = 30  # ajustează după nevoie

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
    for images, labels in loop:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()

        # Mixed precision
        with torch.amp.autocast(device_type='cuda', dtype=torch.float16):
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        loop.set_postfix(loss=running_loss/total, acc=100*correct/total)

    # === Validation per epoch ===
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            with torch.amp.autocast(device_type='cuda', dtype=torch.float16):
                outputs = model(images)
                loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    print(f"Epoch {epoch+1} -> Train Loss: {running_loss/total:.4f}, Train Acc: {100*correct/total:.2f}% | Val Loss: {val_loss/val_total:.4f}, Val Acc: {100*val_correct/val_total:.2f}%")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 195MB/s]
/tmp/ipython-input-1710391906.py:10: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
Epoch 1/30: 100%|██████████| 150/150 [00:39<00:00,  3.82it/s, acc=82.7, loss=0.417]


Epoch 1 -> Train Loss: 0.4175, Train Acc: 82.71% | Val Loss: 0.4388, Val Acc: 83.50%


Epoch 2/30: 100%|██████████| 150/150 [00:38<00:00,  3.86it/s, acc=92.6, loss=0.185]


Epoch 2 -> Train Loss: 0.1848, Train Acc: 92.65% | Val Loss: 0.3140, Val Acc: 88.42%


...

Epoch 28 -> Train Loss: 0.0096, Train Acc: 99.73% | Val Loss: 0.6266, Val Acc: 88.92%


Epoch 29/30: 100%|██████████| 150/150 [00:39<00:00,  3.82it/s, acc=99.6, loss=0.0122]


Epoch 29 -> Train Loss: 0.0122, Train Acc: 99.56% | Val Loss: 0.4566, Val Acc: 91.00%


Epoch 30/30: 100%|██████████| 150/150 [00:38<00:00,  3.94it/s, acc=99.2, loss=0.0235]


Epoch 30 -> Train Loss: 0.0235, Train Acc: 99.23% | Val Loss: 0.4658, Val Acc: 90.17%


In [13]:
torch.save(model.state_dict(), "/content/drive/MyDrive/oct_modelvar2_2_resnet18.pth")